### Try a different execution provider

Once a model is in ONNX format, we can use it with many *execution providers*. In ONNX, an execution provider an interface that lets ONNX models run with special hardware-specific capabilities. Until now, we have been using the `CPUExecutionProvider`, but if we use hardware-specific capabilities, e.g. switch out generic implementations of graph operations for implementations that are optimized for specific hardware, we can execute exactly the same model, much faster.

In [1]:
# runs in jupyter container on node-serve-model
import os
import time
import numpy as np
import torch
import onnx
import onnxruntime as ort
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [5]:
# runs in jupyter container on node-serve-model
# Prepare test dataset
# runs in jupyter container on node-serve-model
# Prepare test dataset
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader,Subset
from transformers import AutoTokenizer

# Load your eval file
df = pd.read_json("/mnt/eval.jsonl", lines=True)

model_id = "google-t5/t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_id)

max_input_len = 512
max_target_len = 256


class RecipeEvalDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_input_len=512, max_target_len=256):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_input_len = max_input_len
        self.max_target_len = max_target_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Keep the "bad" input mostly intact
        input_text = row["input"].strip()
        target_text = row["target"].strip()

        # Optional: remove metadata noise only
        input_text = input_text.replace("<source:web_scrape>", "").strip()

        model_inputs = self.tokenizer(
            input_text,
            max_length=self.max_input_len,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

        labels = self.tokenizer(
            text_target=target_text,
            max_length=self.max_target_len,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

        input_ids = model_inputs["input_ids"].squeeze(0)
        attention_mask = model_inputs["attention_mask"].squeeze(0)
        labels = labels["input_ids"].squeeze(0)

        # Optional: ignore padding tokens in loss
        labels[labels == tokenizer.pad_token_id] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "raw_input": input_text,
            "raw_target": target_text,
        }


dataset = RecipeEvalDataset(df, tokenizer, max_input_len, max_target_len)

dataloader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=False,
)

# from torch.utils.data import Subset, DataLoader

subset_size = 200
small_dataset = Subset(dataset, range(subset_size))

small_dataloader = DataLoader(
    small_dataset,
    batch_size=4,
    shuffle=False,
)

batch = next(iter(small_dataloader))
print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["labels"].shape)
print(batch["raw_input"][0][:200])
print(batch["raw_target"][0][:200])

torch.Size([4, 512])
torch.Size([4, 512])
torch.Size([4, 256])
fix recipe: Title: Valentine's Thins
Ingredients: 36 RITZ Crackers with Whole Wheat | 2 pkg. (4 oz. each) BAKER'S Semi-Sweet Chocolate, broken into pieces, melted | 36 berry-shaped gummy candies (abou
Title: Valentine's Thins
Ingredients: 36 Ritz Crackers with Whole Wheat | 1 pkg. (225 g) Baker's Semi-Sweet Chocolate, melted | 36 Maynards Swedish Berries (about 2 pkg. [64 g each]) | 2 oz. Baker's W


In [6]:
import time
import numpy as np
from tqdm.auto import tqdm
from ignite.metrics import RougeL


def benchmark_onnx_model(onnx_model, dataloader, tokenizer, max_new_tokens=256):
    print("ONNX model loaded with Hugging Face Optimum")

    # ----------------------------
    # 1. ROUGE-L
    # ----------------------------
    predictions = []
    references = []

    rouge_l = RougeL(multiref="best")
    rouge_l.reset()

    for batch in tqdm(dataloader, desc="Evaluating ONNX model"):
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        refs = batch["raw_target"]

        output_ids = onnx_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
        )

        preds = tokenizer.batch_decode(output_ids, skip_special_tokens=True)

        predictions.extend(preds)
        references.extend(refs)

        candidates = [p.split() for p in preds]
        gold_refs = [[r.split()] for r in refs]
        rouge_l.update((candidates, gold_refs))

    rouge_scores = rouge_l.compute()

    print(f"Rouge-L-P: {rouge_scores['Rouge-L-P']:.4f}")
    print(f"Rouge-L-R: {rouge_scores['Rouge-L-R']:.4f}")
    print(f"Rouge-L-F: {rouge_scores['Rouge-L-F']:.4f}")

    # ----------------------------
    # 2. Single-sample latency
    # ----------------------------
    num_trials = 100

    batch = next(iter(dataloader))
    single_input_ids = batch["input_ids"][0].unsqueeze(0)
    single_attention_mask = batch["attention_mask"][0].unsqueeze(0)

    # Warm-up
    _ = onnx_model.generate(
        input_ids=single_input_ids,
        attention_mask=single_attention_mask,
        max_new_tokens=max_new_tokens,
    )

    latencies = []
    for _ in tqdm(range(num_trials), desc="Single-sample latency"):
        start_time = time.perf_counter()
        _ = onnx_model.generate(
            input_ids=single_input_ids,
            attention_mask=single_attention_mask,
            max_new_tokens=max_new_tokens,
        )
        latencies.append(time.perf_counter() - start_time)

    print(f"Inference Latency (single sample, median): {np.percentile(latencies, 50) * 1000:.2f} ms")
    print(f"Inference Latency (single sample, 95th percentile): {np.percentile(latencies, 95) * 1000:.2f} ms")
    print(f"Inference Latency (single sample, 99th percentile): {np.percentile(latencies, 99) * 1000:.2f} ms")
    print(f"Inference Throughput (single sample): {num_trials / np.sum(latencies):.2f} samples/sec")

    # ----------------------------
    # 3. Batch throughput
    # ----------------------------
    num_batches = 50

    batch = next(iter(dataloader))
    batch_input_ids = batch["input_ids"]
    batch_attention_mask = batch["attention_mask"]

    # Warm-up
    _ = onnx_model.generate(
        input_ids=batch_input_ids,
        attention_mask=batch_attention_mask,
        max_new_tokens=max_new_tokens,
    )

    batch_times = []
    for _ in tqdm(range(num_batches), desc="Batch throughput"):
        start_time = time.perf_counter()
        _ = onnx_model.generate(
            input_ids=batch_input_ids,
            attention_mask=batch_attention_mask,
            max_new_tokens=max_new_tokens,
        )
        batch_times.append(time.perf_counter() - start_time)

    batch_fps = (batch_input_ids.shape[0] * num_batches) / np.sum(batch_times)
    print(f"Batch Throughput: {batch_fps:.2f} samples/sec")

/opt/conda/lib/python3.12/site-packages/ignite/handlers/checkpoint.py:16: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import ZeroRedundancyOptimizer


#### CPU execution provider

First, for reference, we’ll repeat our performance test for the (unquantized model with) `CPUExecutionProvider`:

In [3]:
# runs in jupyter container on node-serve-model
# onnx_model_path = "models/food11.onnx"
# ort_session = ort.InferenceSession(onnx_model_path, providers=['CPUExecutionProvider'])
# benchmark_session(ort_session)

import onnxruntime as ort
from optimum.onnxruntime import ORTModelForSeq2SeqLM
from transformers import AutoTokenizer

model_dir = "/home/jovyan/work/models/onnx/onnx"

session_options = ort.SessionOptions()
session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

tokenizer = AutoTokenizer.from_pretrained(model_dir)

onnx_model = ORTModelForSeq2SeqLM.from_pretrained(
    model_dir,
    encoder_file_name="encoder_model.onnx",
    decoder_file_name="decoder_model.onnx",
    decoder_with_past_file_name="decoder_with_past_model.onnx",
    providers=["CPUExecutionProvider"],
    session_options=session_options,
)

In [6]:
benchmark_onnx_model(
    onnx_model=onnx_model,
    dataloader=small_dataloader,
    tokenizer=tokenizer,
    max_new_tokens=256,
)

ONNX model loaded with Hugging Face Optimum


Evaluating ONNX model:   0%|          | 0/50 [00:00<?, ?it/s]

Rouge-L-P: 0.4636
Rouge-L-R: 0.2368
Rouge-L-F: 0.2368


Single-sample latency:   0%|          | 0/100 [00:00<?, ?it/s]

Inference Latency (single sample, median): 611.83 ms
Inference Latency (single sample, 95th percentile): 655.34 ms
Inference Latency (single sample, 99th percentile): 669.45 ms
Inference Throughput (single sample): 1.61 samples/sec


Batch throughput:   0%|          | 0/50 [00:00<?, ?it/s]

Batch Throughput: 3.87 samples/sec


<!--
Execution provider: ['CPUExecutionProvider']
Accuracy: 90.59% (3032/3347 correct)
Inference Latency (single sample, median): 9.93 ms
Inference Latency (single sample, 95th percentile): 14.20 ms
Inference Latency (single sample, 99th percentile): 14.43 ms
Inference Throughput (single sample): 91.10 FPS
Batch Throughput: 1042.47 FPS
-->

#### CUDA execution provider

Before we can use CUDA and TensorRT execution providers, we need to switch from the `jupyter-onnx-base` image to the `jupyter-onnx-gpu` image.

Close this Jupyter server tab - you will reopen it shortly, with a new token.

Go back to your SSH session on “node-serve-model”, and stop the current Jupyter server with:

``` bash
# runs on node-serve-model
docker stop jupyter
```

``` bash 
cd /recipe-scraper-mlops/serving/docker```
Build the GPU image:

``` bash
# runs on node-serve-model
docker build -t jupyter-onnx-gpu -f serve-model-chi/docker/Dockerfile.jupyter-onnx-nvidia .
```

Then launch a new one with the GPU image:

``` bash
# runs on node-serve-model
docker run  -d --rm  -p 8888:8888 \
    --gpus all \
    --shm-size 16G \
    -v ~/recipe-scraper-mlops/serving/workspace:/home/jovyan/work/ \
    -v recipe-nlg:/mnt/ \
    -e RECIPE-NLG-FILE=/mnt/eval.jsonl \
    --name jupyter \
    jupyter-onnx-gpu
```

Then get a new token:

``` bash
# runs on node-serve-model
docker exec jupyter jupyter server list
```

and look for a line like

    http://localhost:8888/?token=XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX

Paste this into a browser tab, but in place of `localhost`, substitute the floating IP assigned to your instance, to open the Jupyter notebook interface that is running *on your compute instance*.

Then, in the file browser on the left side, open the “work” directory and then click on the `8_ep_onnx.ipynb` notebook to continue.

Run the three cells at the top, which `import` libraries, set up the data loaders, and define the `benchmark_session` function. Then continue with CUDA and TensorRT:

Next, we’ll try it with the CUDA execution provider, which will execute the model on the GPU:

#### Install things

In [2]:
!pip install "optimum[onnxruntime]" transformers huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 138.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.1/801.1 kB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 159.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 154.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 135.8 MB/s eta 0:00:00


In [3]:
!pip install pytorch-ignite

#### PyTorch on CUDA

In [1]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("torch cuda version:", torch.version.cuda)

device = torch.device("cuda:0")

model_id = "google-t5/t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(device)
model.eval()

text = "fix recipe: test"
inputs = tokenizer(text, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=8)

print(tokenizer.decode(out[0], skip_special_tokens=True))

torch: 2.5.1+cu124
cuda available: True
torch cuda version: 12.4
fix recipe: test


In [3]:
import time
import numpy as np
import torch
from tqdm.auto import tqdm
from ignite.metrics import RougeL


def benchmark_pytorch_model(model, dataloader, tokenizer, device, max_new_tokens=256):
    model = model.to(device)
    model.eval()

    print(f"Device: {device}")

    # ----------------------------
    # 1. ROUGE-L
    # ----------------------------
    predictions = []
    references = []

    rouge_l = RougeL(multiref="best")
    rouge_l.reset()

    for batch in tqdm(dataloader, desc="Evaluating PyTorch model"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        refs = batch["raw_target"]

        with torch.no_grad():
            output_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=max_new_tokens,
            )

        preds = tokenizer.batch_decode(output_ids, skip_special_tokens=True)

        predictions.extend(preds)
        references.extend(refs)

        candidates = [p.split() for p in preds]
        gold_refs = [[r.split()] for r in refs]
        rouge_l.update((candidates, gold_refs))

    rouge_scores = rouge_l.compute()

    print(f"Rouge-L-P: {rouge_scores['Rouge-L-P']:.4f}")
    print(f"Rouge-L-R: {rouge_scores['Rouge-L-R']:.4f}")
    print(f"Rouge-L-F: {rouge_scores['Rouge-L-F']:.4f}")

    # ----------------------------
    # 2. Single-sample latency
    # ----------------------------
    num_trials = 100

    batch = next(iter(dataloader))
    single_input_ids = batch["input_ids"][0].unsqueeze(0).to(device)
    single_attention_mask = batch["attention_mask"][0].unsqueeze(0).to(device)

    with torch.no_grad():
        _ = model.generate(
            input_ids=single_input_ids,
            attention_mask=single_attention_mask,
            max_new_tokens=max_new_tokens,
        )
    if device.type == "cuda":
        torch.cuda.synchronize()

    latencies = []
    for _ in tqdm(range(num_trials), desc="Single-sample latency"):
        start_time = time.perf_counter()
        with torch.no_grad():
            _ = model.generate(
                input_ids=single_input_ids,
                attention_mask=single_attention_mask,
                max_new_tokens=max_new_tokens,
            )
        if device.type == "cuda":
            torch.cuda.synchronize()
        latencies.append(time.perf_counter() - start_time)

    print(f"Inference Latency (single sample, median): {np.percentile(latencies, 50) * 1000:.2f} ms")
    print(f"Inference Latency (single sample, 95th percentile): {np.percentile(latencies, 95) * 1000:.2f} ms")
    print(f"Inference Latency (single sample, 99th percentile): {np.percentile(latencies, 99) * 1000:.2f} ms")
    print(f"Inference Throughput (single sample): {num_trials / np.sum(latencies):.2f} samples/sec")

    # ----------------------------
    # 3. Batch throughput
    # ----------------------------
    num_batches = 50

    batch = next(iter(dataloader))
    batch_input_ids = batch["input_ids"].to(device)
    batch_attention_mask = batch["attention_mask"].to(device)

    with torch.no_grad():
        _ = model.generate(
            input_ids=batch_input_ids,
            attention_mask=batch_attention_mask,
            max_new_tokens=max_new_tokens,
        )
    if device.type == "cuda":
        torch.cuda.synchronize()

    batch_times = []
    for _ in tqdm(range(num_batches), desc="Batch throughput"):
        start_time = time.perf_counter()
        with torch.no_grad():
            _ = model.generate(
                input_ids=batch_input_ids,
                attention_mask=batch_attention_mask,
                max_new_tokens=max_new_tokens,
            )
        if device.type == "cuda":
            torch.cuda.synchronize()
        batch_times.append(time.perf_counter() - start_time)

    batch_fps = (batch_input_ids.shape[0] * num_batches) / np.sum(batch_times)
    print(f"Batch Throughput: {batch_fps:.2f} samples/sec")

/opt/conda/lib/python3.12/site-packages/ignite/handlers/checkpoint.py:16: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import ZeroRedundancyOptimizer


In [6]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

benchmark_pytorch_model(
    model=model,
    dataloader=small_dataloader,
    tokenizer=tokenizer,
    device=device,
    max_new_tokens=256,
)

Device: cuda:0


Evaluating PyTorch model:   0%|          | 0/50 [00:00<?, ?it/s]

Rouge-L-P: 0.4636
Rouge-L-R: 0.2368
Rouge-L-F: 0.2368


Single-sample latency:   0%|          | 0/100 [00:00<?, ?it/s]

Inference Latency (single sample, median): 1275.68 ms
Inference Latency (single sample, 95th percentile): 1296.56 ms
Inference Latency (single sample, 99th percentile): 1299.55 ms
Inference Throughput (single sample): 0.78 samples/sec


Batch throughput:   0%|          | 0/50 [00:00<?, ?it/s]

Batch Throughput: 3.00 samples/sec


<!--
Execution provider: ['CUDAExecutionProvider', 'CPUExecutionProvider']
Accuracy: 90.59% (3032/3347 correct)
Inference Latency (single sample, median): 0.89 ms
Inference Latency (single sample, 95th percentile): 0.90 ms
Inference Latency (single sample, 99th percentile): 0.91 ms
Inference Throughput (single sample): 1117.06 FPS
Batch Throughput: 5181.99 FPS
-->

#### TensorRT execution provider

The TensorRT execution provider will optimize the model for inference on NVIDIA GPUs. It will take a long time to run this cell, because it spends a lot of time optimizing the model (finding the best subgraphs, etc.) - but once the model is loaded, its inference time will be much faster than any of our previous tests.

In [8]:
!pip install torch-tensorrt tensorrt

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... one
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 96.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 147.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 169.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 131.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 181.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 182.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 93.7 MB/s eta 0:00:0000:0100:01

In [10]:
import sys
!{sys.executable} -m pip show torch torch-tensorrt tensorrt tensorrt-cu12 tensorrt-cu12-bindings tensorrt-cu12-libs

Name: torch
Version: 2.11.0
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License: BSD-3-Clause
Location: /opt/conda/lib/python3.12/site-packages
Requires: cuda-bindings, cuda-toolkit, filelock, fsspec, jinja2, networkx, nvidia-cudnn-cu13, nvidia-cusparselt-cu13, nvidia-nccl-cu13, nvidia-nvshmem-cu13, setuptools, sympy, triton, typing-extensions
Required-by: optimum, pytorch-ignite, torch_tensorrt, torchaudio, torchvision
---
Name: torch_tensorrt
Version: 2.11.0
Summary: Torch-TensorRT is a package which allows users to automatically compile PyTorch and TorchScript modules to TensorRT while remaining in PyTorch
Home-page: https://pytorch.org/tensorrt
Author: 
Author-email: NVIDIA Corporation <narens@nvidia.com>
License: Copyright (c) 2020-present, NVIDIA CORPORATION. All rights reserved.
Copyright (c) Meta Platforms, Inc. and affiliates.


Redistribution and u

In [9]:
# runs in jupyter container on node-serve-model
import torch_tensorrt

optimized_model = torch.compile(model, backend="tensorrt")
# onnx_model_path = "models/food11.onnx"
# ort_session = ort.InferenceSession(onnx_model_path, providers=['TensorrtExecutionProvider'])
# benchmark_session(ort_session)
# ort.get_device()


Neither TRTLLM_PLUGIN_PATH is set nor is it directed to download the shared library. Please set either of the two to use TRT-LLM libraries in torchTRT


[04/06/2026-20:39:18] [TRT] [W] Functionality provided through tensorrt.plugin module is experimental.


OSError: libcudart.so.13: cannot open shared object file: No such file or directory

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

benchmark_pytorch_model(
    model=optimized_model,
    dataloader=small_dataloader,
    tokenizer=tokenizer,
    device=device,
    max_new_tokens=256,
)

<!--
Execution provider: ['TensorrtExecutionProvider', 'CPUExecutionProvider']
Accuracy: 90.59% (3032/3347 correct)
Inference Latency (single sample, median): 0.63 ms
Inference Latency (single sample, 95th percentile): 0.64 ms
Inference Latency (single sample, 99th percentile): 0.70 ms
Inference Throughput (single sample): 1572.61 FPS
Batch Throughput: 9274.45 FPS
-->

#### OpenVINO execution provider

Even just on CPU, we can still use an optimized execution provider to improve inference performance. We will try out the Intel [OpenVINO](https://github.com/openvinotoolkit/openvino) execution provider. However, ONNX runtime can be built to support CUDA/TensorRT or OpenVINO, but not both at the same time, so we will need to bring up a new container.

Close this Jupyter server tab - you will reopen it shortly, with a new token.

Go back to your SSH session on “node-serve-model”, and stop the current Jupyter server:

``` bash
# runs on node-serve-model
docker stop jupyter
```

Build the OpenVINO image:

``` bash
# runs on node-serve-model
docker build -t jupyter-onnx-openvino -f serve-model-chi/docker/Dockerfile.jupyter-onnx-openvino .
```

Then, launch a container with the OpenVINO image:

``` bash
# runs on node-serve-model
docker run  -d --rm  -p 8888:8888 \
    --shm-size 16G \
    -v ~/serve-model-chi/workspace:/home/jovyan/work/ \
    -v food11:/mnt/ \
    -e FOOD11_DATA_DIR=/mnt/Food-11 \
    --name jupyter \
    jupyter-onnx-openvino
```

To access the Jupyter service, we will need its randomly generated secret token (which secures it from unauthorized access).

Run

``` bash
# runs on node-serve-model
docker exec jupyter jupyter server list
```

and look for a line like

    http://localhost:8888/?token=XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX

Paste this into a browser tab, but in place of `localhost`, substitute the floating IP assigned to your instance, to open the Jupyter notebook interface that is running *on your compute instance*.

Then, in the file browser on the left side, open the “work” directory and then click on the `8_ep_onnx.ipynb` notebook to continue.

Run the three cells at the top, which `import` libraries, set up the data loaders, and define the `benchmark_session` function. Then, skip to the OpenVINO section and run:

In [ ]:
# runs in jupyter container on node-serve-model
onnx_model_path = "models/food11.onnx"
ort_session = ort.InferenceSession(onnx_model_path, providers=['OpenVINOExecutionProvider'])
benchmark_session(ort_session)
ort.get_device()

<!--

On AMD EPYC

Execution provider: ['OpenVINOExecutionProvider', 'CPUExecutionProvider']
Accuracy: 90.59% (3032/3347 correct)
Inference Latency (single sample, median): 1.39 ms
Inference Latency (single sample, 95th percentile): 1.89 ms
Inference Latency (single sample, 99th percentile): 1.92 ms
Inference Throughput (single sample): 646.63 FPS
Batch Throughput: 1624.30 FPS

On Intel

Execution provider: ['OpenVINOExecutionProvider', 'CPUExecutionProvider']
Accuracy: 90.59% (3032/3347 correct)
Inference Latency (single sample, median): 1.55 ms
Inference Latency (single sample, 95th percentile): 1.76 ms
Inference Latency (single sample, 99th percentile): 1.81 ms
Inference Throughput (single sample): 663.72 FPS
Batch Throughput: 2453.48 FPS

-->

When you are done, download the fully executed notebook from the Jupyter container environment for later reference. (Note: because it is an executable file, and you are downloading it from a site that is not secured with HTTPS, you may have to explicitly confirm the download in some browsers.)